# Embedding Analysis

Embed item titles with SBERT (`all-MiniLM-L6-v2`) and use them to find similar products. Inputs: `data/df_features.pkl` (output of `feature_extraction_workflow/extract_features.ipynb`). Output: `data/df_features_with_embeddings.pkl` (gitignored).

Sections:
1. Load data
2. Generate SBERT embeddings for `title_cleaned`
3. Save the dataframe with embeddings
4. Build the normalized matrix
5. Run each similarity helper on a sample ASIN
6. PCA visualizations

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

from embedding_analysis import (
    build_normalized_matrix,
    create_embeddings,
    get_similar_items_by_category,
    get_similar_items_diff_cat3,
    get_similar_items_diff_cat4_product_type,
    get_similar_items_same_cat3,
    get_similar_items_same_cat3_diff_cat4,
    get_similar_items_same_cat3_diff_cat4_product_type,
    get_top_n_similar,
    visualize_pca_overall,
    visualize_pca_per_cat2,
)

pd.options.display.max_colwidth = None
pd.options.display.max_columns = None

## 1. Load data

Read the feature dataframe produced by `extract_features.ipynb`.

In [2]:
DATA_DIR = PROJECT_ROOT / 'data'
FEATURES_PATH = DATA_DIR / 'df_features.pkl'
EMBEDDINGS_PATH = DATA_DIR / 'df_features_with_embeddings.pkl'

df_features = pd.read_pickle(FEATURES_PATH)
print(f'Loaded {len(df_features):,} rows x {df_features.shape[1]} cols from {FEATURES_PATH}')

needed = ['asin', 'title', 'title_cleaned', 'cat_2', 'cat_3', 'cat_4', 'Product_Type']
for col in needed:
    print(f'  {col:20s} present: {col in df_features.columns}')

Loaded 1,134,566 rows x 81 cols from /Users/lazr/PycharmProjects/RecSystem/data/df_features.pkl
  asin                 present: True
  title                present: True
  title_cleaned        present: True
  cat_2                present: True
  cat_3                present: True
  cat_4                present: True
  Product_Type         present: True


## 2. Generate SBERT embeddings

`all-MiniLM-L6-v2` (small SBERT, 384-dim) on `title_cleaned`. Encoding ~1.1M titles on CPU takes a while — set `show_progress_bar=True` to monitor. If `data/df_features_with_embeddings.pkl` already exists, the next cell short-circuits and loads it.

In [ ]:
if EMBEDDINGS_PATH.exists():
    df_emb = pd.read_pickle(EMBEDDINGS_PATH)
    print(f'Loaded existing embeddings from {EMBEDDINGS_PATH}')
    print(f'Shape: {df_emb.shape}')
else:
    df_emb = create_embeddings(
        df_features,
        text_col='title_cleaned',
        model_name='all-MiniLM-L6-v2',
        batch_size=256,
        embedding_col='title_embedding',
    )
    print(f'\nResulting dataframe: {df_emb.shape}')

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5159.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 1,134,566 rows from 'title_cleaned' with all-MiniLM-L6-v2...


Batches:   2%|▏         | 76/4432 [00:21<14:51,  4.88it/s] 

## 3. Save

Persist to `data/df_features_with_embeddings.pkl`. Path is in `.gitignore` so it won't be committed.

In [ ]:
if not EMBEDDINGS_PATH.exists():
    df_emb.to_pickle(EMBEDDINGS_PATH)
    print(f'Saved {len(df_emb):,} rows to {EMBEDDINGS_PATH}')
else:
    print(f'Already exists at {EMBEDDINGS_PATH}; skipping write.')

## 4. Build the normalized matrix

Stack the embedding column and L2-normalize each row, so cosine similarity becomes a dot product. `asins` is a parallel array used by every helper to look up rows by ASIN.

In [ ]:
matrix_norm, asins = build_normalized_matrix(df_emb, embedding_col='title_embedding')
print(f'matrix_norm shape: {matrix_norm.shape}')
print(f'asins length     : {len(asins):,}')
print(f'first row L2 norm: {np.linalg.norm(matrix_norm[0]):.4f} (should be 1.0)')

## 5. Similarity helpers

Each helper takes one query ASIN, scores it against the whole matrix (or a category-filtered subset), and returns the top-N as a small dataframe. Replace `QUERY_ASIN` with any ASIN present in `asins`.

In [ ]:
QUERY_ASIN = 'B00029TCRG'   # same example used in analyze_features_eda.ipynb
N = 10

if QUERY_ASIN not in set(asins):
    QUERY_ASIN = asins[0]
    print(f'Default ASIN not in this dataset; falling back to first row: {QUERY_ASIN}')
else:
    print(f'Using query ASIN: {QUERY_ASIN}')

In [ ]:
# 5.1 Plain top-N (no filter, no boost)
get_top_n_similar(QUERY_ASIN, df_emb, matrix_norm, asins, n=N)

In [ ]:
# 5.2 Top-N with cat_3 (+0.10) and cat_4 (+0.05) score boosts
get_similar_items_by_category(QUERY_ASIN, df_emb, matrix_norm, asins, n=N)

In [ ]:
# 5.3 Restricted to the same cat_3 as the query
get_similar_items_same_cat3(QUERY_ASIN, df_emb, matrix_norm, asins, n=N)

In [ ]:
# 5.4 Same cat_3, different cat_4
get_similar_items_same_cat3_diff_cat4(QUERY_ASIN, df_emb, matrix_norm, asins, n=N)

In [ ]:
# 5.5 Same cat_3, different cat_4 AND different Product_Type
get_similar_items_same_cat3_diff_cat4_product_type(QUERY_ASIN, df_emb, matrix_norm, asins, n=N)

In [ ]:
# 5.6 Different cat_4 AND different Product_Type (any cat_3)
get_similar_items_diff_cat4_product_type(QUERY_ASIN, df_emb, matrix_norm, asins, n=N)

In [ ]:
# 5.7 Items NOT in the same cat_3 as the query
get_similar_items_diff_cat3(QUERY_ASIN, df_emb, matrix_norm, asins, n=N)

## 6. PCA visualizations

Reduce the 384-dim embeddings to 2D and scatter colored by category. The overall view captures cat_2-level separation; the per-cat_2 views zoom into one group at a time and color by cat_3.

**Heads up**: PCA on a ~1M × 384 matrix takes a minute or so, and rendering ~1M scatter points is heavy. If you're iterating, sample the dataframe first (e.g. `df_sample = df_emb.sample(50_000, random_state=42)` then rebuild the matrix on the sample).

In [ ]:
# 6.1 Overall PCA, colored by cat_2
visualize_pca_overall(matrix_norm, df_emb, color_by='cat_2')

In [ ]:
# 6.2 One PCA scatter per cat_2 value, colored by cat_3
visualize_pca_per_cat2(matrix_norm, df_emb)